In [2]:
import os
os.mkdir("Na_Na3SbS4_Data")

FileExistsError: [Errno 17] File exists: 'Na_Na3SbS4_Data'

In [3]:
import os
from mp_api.client import MPRester

# 1. 设置输出目录
# 将目录名改为更相关的名称
output_dir = "Na_Na3SbS4_Data/structures"
os.makedirs(output_dir, exist_ok=True)

# 2. 查询 Materials Project
# 为了安全，请将下面引号中的内容替换为你自己的 API Key
# 注意：不要在公开场合（如论坛、GitHub）泄露你的真实 Key
with MPRester("bTeurW7C1cBsIosJYTS5jW0LY7XJ8CCt") as mpr:
    
    # --- 搜索 1: 纯金属 Na (阳极) ---
    print("正在搜索纯 Na (基态及亚稳态)...")
    # energy_above_hull 设置为 0.05 eV/atom 以内，确保包含 BCC 结构或其他可能的相
    docs_Na = mpr.materials.summary.search(
        chemsys="Na", 
        energy_above_hull=(0, 0.05)
    )
    
    # --- 搜索 2: Na3SbS4 (固态电解质) ---
    print("正在搜索 Na3SbS4 (所有构型)...")
    # 直接使用 formula 搜索，可以找到立方相(cubic)和四方相(tetragonal)
    docs_Elyte = mpr.materials.summary.search(
        formula="Na3SbS4", 
        energy_above_hull=(0, 0.1) # 放宽一点范围以包含常见的亚稳相
    )

    # --- (可选) 搜索 3: 潜在的界面反应产物 ---
    # 如果你要研究腐蚀，界面通常会生成 Na2S 和 Na3Sb
    # 如果需要，取消下面的注释即可下载这些结构
    # print("正在搜索潜在反应产物 (Na2S, Na3Sb)...")
    # docs_products = mpr.materials.summary.search(formula=["Na2S", "Na3Sb"], energy_above_hull=(0, 0.05))
    
    # 合并结果 (这里暂时只合并 Na 和 Na3SbS4)
    all_docs = docs_Na + docs_Elyte 
    # 如果开启了上面的 docs_products，请改成: all_docs = docs_Na + docs_Elyte + docs_products
    
    print(f"总共找到 {len(all_docs)} 个结构。")

    # 3. 保存结构
    for doc in all_docs:
        # 提取结构对象 (Pymatgen Structure)
        structure = doc.structure
        material_id = doc.material_id
        formula = doc.formula_pretty
        
        # 定义文件名: 例如 "Na3SbS4_mp-9999.vasp"
        # 使用 .vasp 后缀，直接对应 POSCAR 格式，方便 VASP/DeepMD 使用
        filename = f"{formula}_{material_id}.vasp"
        file_path = os.path.join(output_dir, filename)
        
        # 保存文件
        # fmt="poscar" 会生成 VASP 格式 (即 POSCAR)
        structure.to(filename=file_path, fmt="poscar")
        print(f"已保存: {filename}")

print(f"\n所有结构已存储在 '{output_dir}' 目录下。")

/dssg/home/acct-matxzl/matxzl/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在搜索纯 Na (基态及亚稳态)...


Retrieving SummaryDoc documents: 100%|██████████| 13/13 [00:00<00:00, 315178.91it/s]


正在搜索 Na3SbS4 (所有构型)...


Retrieving SummaryDoc documents: 100%|██████████| 1/1 [00:00<00:00, 31068.92it/s]

总共找到 14 个结构。
已保存: Na_mp-10172.vasp
已保存: Na_mp-1186040.vasp
已保存: Na_mp-1186055.vasp
已保存: Na_mp-1186081.vasp
已保存: Na_mp-127.vasp
已保存: Na_mp-1525464.vasp
已保存: Na_mp-1545923.vasp
已保存: Na_mp-2018774.vasp
已保存: Na_mp-567772.vasp
已保存: Na_mp-973198.vasp
已保存: Na_mp-974558.vasp
已保存: Na_mp-974920.vasp
已保存: Na_mp-982370.vasp
已保存: Na3SbS4_mp-10167.vasp

所有结构已存储在 'Na_Na3SbS4_Data/structures' 目录下。


In [1]:
!ls

dpdispatcher.log  LiGa	main-3.ipynb  Na_Na3SbS4_Data  vasp-Na3SbS4.ipynb


In [6]:
import os
import warnings
from pymatgen.core import Structure
from pymatgen.io.vasp.sets import MPRelaxSet
import math
# 1. Setup Directories
# input_dir = "LiGa/structures"  <-- 原代码
input_dir = "Na_Na3SbS4_Data/structures" # <-- 修改后

# base_output_dir = "LiGa/AIMD_runs" <-- 原代码
base_output_dir = "Na_Na3SbS4_Data/AIMD_runs" # <-- 修改后  # Separate folder to keep things clean
os.makedirs(base_output_dir, exist_ok=True)

# 2. Define AIMD Settings (Overrides standard RelaxSet)
# These settings ensure we sample the potential energy surface, not just find the minimum.
aimd_settings = {
    # --- 基础 MD 设置 ---
    "IBRION": 0,          # MD 模式
    "NSW": 2000,          # 步数 (视计算资源而定，界面建议跑久点)
    "POTIM": 2.0,         # 时间步长 2.0 fs (如果 Na 乱飞导致报错，可降为 1.5)
    "TEBEG": 800,         # 起始温度
    "TEEND": 800,         # 结束温度
    
    # --- 关键修改区 ---
    "ISYM": 0,            # 关闭对称性 (必须)
    "SMASS": 0,           # Nose-Hoover 热浴 (NVT 系综)
    "KBLOCK": 1,          # [重要修改] 每 1 步保存一次 XDATCAR，确保拿到所有训练数据
    "NELM": 100,          # [新增] 电子步最大迭代次数，防止高温不收敛
    
    # --- 电子结构与精度 ---
    "ALGO": "Fast",       # 推荐算法
    "PREC": "Normal",     # 精度 Normal 足够
    "ISMEAR": 0,          # Gaussian Smearing (MD 推荐)
    "SIGMA": 0.1,         # [重要修改] 增大到 0.1 eV 以适应金属 Na 的收敛
    "LREAL": "Auto",      # [新增] 投影算符自动，大体系(>50原子)能显著加速计算
    
    # --- 晶胞控制 ---
    "ISIF": 2,            # 固定体积。
                          # 界面计算尽量不要用 NPT (ISIF=3)，容易造成盒子变形、爆炸。
                          # 建议手动改变晶格常数来模拟不同压力。
    
    # --- 输出控制 ---
    "LWAVE": False,       # 不存波函数，省空间
    "LCHARG": False,      # 不存电荷密度
    "NCORE": 8,           # 并行核数 (请根据你的机器实际核心数调整，例如单节点64核可设为8或16)
}

# 3. Processing Loop
print(f"Reading structures from {input_dir}...")

for filename in os.listdir(input_dir):
    if filename.endswith(".cif") or filename.endswith(".vasp"):
        file_path = os.path.join(input_dir, filename)
        struct_name = os.path.splitext(filename)[0]
        
        try:
            # Load Structure
            structure = Structure.from_file(file_path)
            
            # --- CRITICAL FOR MLIP: SUPERCELL GENERATION ---
            # MLIPs have a cutoff radius (usually 4-6 Angstroms).
            # If the cell is smaller than 2x Cutoff, atoms see themselves.
            # We enforce a minimum image distance > 10 Angstroms.
            # --- FIX: Manual Supercell Calculation ---
            # We want the minimum dimension to be at least 10.0 Angstroms
            # to avoid self-interaction artifacts in MACE.
            min_length = 10.0
            
            # Get current lattice lengths (a, b, c)
            lengths = structure.lattice.abc
            
            # Calculate scaling factors: ceil(10.0 / length)
            # Example: if length is 3.0, scaling is ceil(3.33) = 4
            scaling_matrix = [max(1, int(math.ceil(min_length / l))) for l in lengths]
            
            # Apply the supercell
            structure.make_supercell(scaling_matrix)
            
            # Create Output Subdirectory
            task_dir = os.path.join(base_output_dir, struct_name)
            os.makedirs(task_dir, exist_ok=True)
            
            # Generate VASP Inputs
            # We use MPRelaxSet as a base because it handles POTCARs/KPOINTS well,
            # but we strictly override the INCAR for MD.
            vis = MPRelaxSet(
                structure, 
                user_incar_settings=aimd_settings,
                user_kpoints_settings={"reciprocal_density": 50},
                user_potcar_functional="PBE"
            )
            
            # Write input files
            vis.write_input(task_dir)
            print(f"Generated inputs for: {struct_name} (Supercell: {structure.num_sites} atoms)")
            
        except Exception as e:
            print(f"Skipped {filename}: {e}")

print(f"\nDone. VASP inputs are ready in '{base_output_dir}'.")

Reading structures from Na_Na3SbS4_Data/structures...
Generated inputs for: Na_mp-974558 (Supercell: 27 atoms)
Generated inputs for: Na_mp-1186055 (Supercell: 160 atoms)
Generated inputs for: Na_mp-1525464 (Supercell: 64 atoms)
Generated inputs for: Na_mp-2018774 (Supercell: 36 atoms)
Generated inputs for: Na_mp-974920 (Supercell: 27 atoms)
Generated inputs for: Na_mp-973198 (Supercell: 54 atoms)
Generated inputs for: Na_mp-1186040 (Supercell: 64 atoms)
Generated inputs for: Na_mp-567772 (Supercell: 64 atoms)
Generated inputs for: Na3SbS4_mp-10167 (Supercell: 64 atoms)
Generated inputs for: Na_mp-982370 (Supercell: 36 atoms)
Generated inputs for: Na_mp-1545923 (Supercell: 64 atoms)
Generated inputs for: Na_mp-127 (Supercell: 27 atoms)
Generated inputs for: Na_mp-10172 (Supercell: 36 atoms)
Generated inputs for: Na_mp-1186081 (Supercell: 29 atoms)

Done. VASP inputs are ready in 'Na_Na3SbS4_Data/AIMD_runs'.


In [3]:
import os
import glob
from dpdispatcher import Machine, Resources, Task, Submission

# 1. Define the Machine
machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

# 2. Define Resources
# --- 修改点 1: 针对 AIMD 的资源调整 ---
resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=1,           # AIMD 很慢，保持 1 个任务对应 1 个 Slurm Job 比较安全
    module_list=["vasp/6.3.0-intel-2021.4.0"],
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn", # 确认邮箱是否正确
        "#SBATCH --ntasks=64",
        "#SBATCH --time=48:00:00",      # [建议] AIMD 耗时久，显式申请 48 小时（根据你队列的上限调整）
        "#SBATCH --job-name=Na_AIMD"    # [建议] 给任务起个名字，方便在 squeue 中查看
    ]
)


setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)

# 3. Define the Command
run_cmd = "mpirun -n 64 vasp_std"

command = f"{setup_env} && {run_cmd}"

# 4. Collect Tasks
task_list = []

# --- 修改点 2: 更改工作目录路径 ---
# 必须与你之前生成 VASP 输入文件的目录保持一致
work_base = "Na_Na3SbS4_Data/AIMD_runs" 

search_pattern = os.path.join(work_base, "*")

print(f"Scanning {work_base} for tasks...")

for folder_path in glob.glob(search_pattern):
    if os.path.isdir(folder_path):
        folder_name = os.path.basename(folder_path)
        
        task = Task(
            command=command,
            task_work_path=folder_name,
            forward_files=[], 
            # 这里的 backward_files 只是 dpdispatcher 检查任务是否完成的依据
            # 它不会自动把文件传回本地（因为你是 LazyLocal 模式）
            backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR', 'XDATCAR'] 
        )
        task_list.append(task)

print(f"Found {len(task_list)} tasks.")

# 5. Create and Run Submission
if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )

    print(f"Submitting {len(task_list)} tasks to Slurm...")
    submission.run_submission()
    print("Submission finished.")
else:
    print(f"No subdirectories found in {work_base}")
    print("请检查路径是否正确，或是否已运行生成脚本。")

Scanning Na_Na3SbS4_Data/AIMD_runs for tasks...
Found 5 tasks.
Submitting 5 tasks to Slurm...
2025-12-09 14:19:25,809 - INFO : info:check_all_finished: False
2025-12-09 14:19:25,885 - INFO : job: 99252af3d6b404c7ba239ccccd246ecf11ac7f68 submit; job_id is 50711486
2025-12-09 14:19:25,904 - INFO : job: 28813827240f516f24be7c13e2763b4c2ea65585 submit; job_id is 50711487
2025-12-09 14:19:25,932 - INFO : job: 30c1134fe2e397483c1d6aba6167633740376724 submit; job_id is 50711488
2025-12-09 14:19:25,949 - INFO : job: e90bde71dcc413ce0e51910fef56595ef8a0c944 submit; job_id is 50711489
2025-12-09 14:19:25,967 - INFO : job: 8bf99b59f96235e1d96a86855472fc895ecaf481 submit; job_id is 50711490


In [3]:
import os
import glob
import numpy as np
from ase.io import read, write

In [5]:
import os
from pymatgen.core import Structure, Lattice
from pymatgen.io.vasp.sets import MPRelaxSet
from pymatgen.io.vasp.inputs import Kpoints  # 新增引用

# ================= 配置 =================
output_dir = "Na_Na3SbS4_Data/E0_calcs_Corrected" # 建议换个新文件夹
os.makedirs(output_dir, exist_ok=True)

elements = ["Na", "Sb", "S"]

# 定义每个元素的初始磁矩 (关键修改点)
# Na: 1个单电子, S: 2个, Sb: 3个
magmom_map = {
    "Na": 1,
    "S":  2,
    "Sb": 3
}

# 基础 INCAR 参数
base_incar_settings = {
    "IBRION": -1,        # 静态计算
    "NSW": 0,            # 0 步
    "ISMEAR": 0,         # Gaussian Smearing
    "SIGMA": 0.01,       # [修改] 孤立原子展宽要小，0.05也可以，但0.01更准
    "ISPIN": 2,          # 开启自旋
    "LREAL": False,      # 必须 False
    "LWAVE": False,
    "LCHARG": False,
    "ALGO": "Normal",    
    "NELM": 100,
    "ISIF": 2,
    "ISYM": 0,           # [新增] 关闭对称性，防止轨道强制简并！
    "Ldipol": False      # 确保不开启偶极校正
}

print(f"🚀 开始生成 *修正版* E0 输入文件至: {output_dir}")

for el in elements:
    task_dir = os.path.join(output_dir, el)
    os.makedirs(task_dir, exist_ok=True)
    
    # -------------------------------------------------------
    # 修改点 1: 结构偏心 (打破几何对称性)
    # 不要用 [0.5, 0.5, 0.5]，改用 [0.51, 0.52, 0.53]
    # -------------------------------------------------------
    struct = Structure(
        Lattice.cubic(15.0), 
        [el], 
        [[0.51, 0.52, 0.53]] 
    )
    
    # -------------------------------------------------------
    # 修改点 2: 动态更新 INCAR 加入 MAGMOM
    # -------------------------------------------------------
    # 复制一份基础配置
    user_incar = base_incar_settings.copy()
    # 写入该元素特定的磁矩
    # 正确写法：加上中括号，变成列表
    user_incar["MAGMOM"] = [ magmom_map[el] ]
    
    try:
        # 生成 VASP Set
        vis = MPRelaxSet(
            struct, 
            user_incar_settings=user_incar,
            user_potcar_functional="PBE"
        )
        
        # 写入 INCAR, POSCAR, POTCAR
        vis.write_input(task_dir)
        
        # -------------------------------------------------------
        # 修改点 3: 强制覆盖为 Gamma 点 (1x1x1)
        # MPRelaxSet 默认可能会生成比较密的 K 点，孤立原子不需要
        # -------------------------------------------------------
        kpts = Kpoints.gamma_automatic()
        kpts.write_file(os.path.join(task_dir, "KPOINTS"))
        
        print(f"✅ [成功] {el} -> {task_dir} (MAGMOM={magmom_map[el]})")
        
    except Exception as e:
        print(f"❌ [失败] {el}: {e}")
        if "POTCAR" in str(e):
            print("   (请检查 pymatgen 的 POTCAR 配置)")

print("\n生成完毕！请使用 mpirun -n 16 运行这些任务。")

🚀 开始生成 *修正版* E0 输入文件至: Na_Na3SbS4_Data/E0_calcs_Corrected
❌ [失败] Na: 'list' object has no attribute 'get'
❌ [失败] Sb: 'list' object has no attribute 'get'
❌ [失败] S: 'list' object has no attribute 'get'

生成完毕！请使用 mpirun -n 16 运行这些任务。


In [4]:
import os
import glob
from dpdispatcher import Machine, Resources, Task, Submission

# ================= 配置 =================
# 指向刚才生成 E0 文件的目录
work_base = "Na_Na3SbS4_Data/E0_calcs_Corrected" 

# ================= 1. Define Machine =================
machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

# ================= 2. Define Resources =================
# 使用你验证过的成功配置
resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=3,              # [优化] E0计算很快，把3个任务打包进一个Job，省得排队3次
    module_list=["vasp/6.3.0-intel-2021.4.0"],
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn", 
        "#SBATCH --ntasks=64",
        "#SBATCH --time=01:00:00",      # [修改] E0 计算非常快(几秒钟)，1小时绰绰有余
        "#SBATCH --job-name=E0_Calc"    # [修改] 任务名
    ]
)

# 环境变量设置 (保持你成功的配置)
setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)

# ================= 3. Define Command =================
# 使用绝对路径比较稳妥，或者如果你之前的 'vasp_std' 能跑通也可以
# 建议这里填入你之前找到的绝对路径，例如:
# vasp_exe = "/dssg/opt/.../vasp_std" 
# run_cmd = f"mpirun -n 64 {vasp_exe}"

# 如果你确定直接用 vasp_std 没问题：
run_cmd = "mpirun -n 16 vasp_std"

command = f"{setup_env} && {run_cmd}"

# ================= 4. Collect Tasks =================
task_list = []
search_pattern = os.path.join(work_base, "*")

print(f"Scanning {work_base} for tasks...")

for folder_path in glob.glob(search_pattern):
    if os.path.isdir(folder_path):
        folder_name = os.path.basename(folder_path)
        
        # 这里的 backward_files 很重要，我们要读取 OUTCAR 里的能量
        task = Task(
            command=command,
            task_work_path=folder_name,
            forward_files=[], 
            backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR'] 
        )
        task_list.append(task)

print(f"Found {len(task_list)} tasks.")

# ================= 5. Submit =================
if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )

    print(f"Submitting {len(task_list)} tasks to Slurm...")
    submission.run_submission()
    print("Submission finished. 请等待任务完成 (通常只需几分钟)。")
else:
    print(f"No subdirectories found in {work_base}")

Scanning Na_Na3SbS4_Data/E0_calcs_Corrected for tasks...
Found 3 tasks.
Submitting 3 tasks to Slurm...
2025-12-12 09:22:08,180 - INFO : info:check_all_finished: False
2025-12-12 09:22:08,408 - INFO : job: 94e579f7489664f87e3c4fae1451c5de0942963a submit; job_id is 50788746
2025-12-12 09:22:40,377 - INFO : job: 94e579f7489664f87e3c4fae1451c5de0942963a 50788746 terminated; fail_cout is 1; resubmitting job
2025-12-12 09:22:40,417 - INFO : job:94e579f7489664f87e3c4fae1451c5de0942963a re-submit after terminated; new job_id is 50788748
2025-12-12 09:22:40,639 - INFO : job:94e579f7489664f87e3c4fae1451c5de0942963a job_id:50788748 after re-submitting; the state now is <JobStatus.waiting: 2>
2025-12-12 09:23:40,860 - INFO : job: 94e579f7489664f87e3c4fae1451c5de0942963a 50788748 terminated; fail_cout is 2; resubmitting job
2025-12-12 09:23:40,893 - INFO : job:94e579f7489664f87e3c4fae1451c5de0942963a re-submit after terminated; new job_id is 50788752
2025-12-12 09:23:41,114 - INFO : job:94e579f7489

RuntimeError: Meet errors will handle unexpected submission state.
Debug information: remote_root==/dssg/home/acct-matxzl/matxzl/QiuQizhi/vasp_learn/Na_Na3SbS4_Data/E0_calcs_Corrected.
Debug information: submission_hash==ad1b807d8403d1e5dcb4818ca9ed50ac313cb967.
Please check error messages above and in remote_root. The submission information is saved in /dssg/home/acct-matxzl/matxzl/.dpdispatcher/submission/ad1b807d8403d1e5dcb4818ca9ed50ac313cb967.json.
For furthur actions, run the following command with proper flags: dpdisp submission ad1b807d8403d1e5dcb4818ca9ed50ac313cb967

In [1]:
import os
from pymatgen.core import Structure, Lattice
from pymatgen.io.vasp.sets import MPRelaxSet

# ================= 配置 =================
output_dir = "Na_Na3SbS4_Data/E0_calcs"
os.makedirs(output_dir, exist_ok=True)

# 定义你的体系中的元素
elements = ["Na", "Sb", "S"]

# 定义 E0 计算专用的 INCAR 参数
# 必须与 AIMD 使用相同的泛函 (PBE)，但不能跑动力学
e0_incar_settings = {
    "IBRION": -1,        # 静态计算 (不移动离子)
    "NSW": 0,            # 0 步
    "ISMEAR": 0,         # Gaussian Smearing
    "SIGMA": 0.05,       # 展宽
    "ISPIN": 2,          # 开启自旋 (孤立原子通常有磁矩)
    "LREAL": False,      # [关键] 孤立原子必须关掉实空间投影，否则在大盒子中精度不够
    "LWAVE": False,
    "LCHARG": False,
    "ALGO": "Normal",    # 电子步算法
    "NELM": 100,
    "ISIF": 2            # 固定体积
}

print(f"🚀 开始生成 E0 输入文件至: {output_dir}")

for el in elements:
    task_dir = os.path.join(output_dir, el)
    os.makedirs(task_dir, exist_ok=True)
    
    # 1. 创建结构：15x15x15 的大真空盒子，原子放在正中间
    # 这样可以忽略原子间的相互作用，模拟“孤立”状态
    struct = Structure(
        Lattice.cubic(15.0), 
        [el], 
        [[0.5, 0.5, 0.5]]
    )
    
    try:
        # 2. 生成 VASP 文件 (INCAR, POSCAR, KPOINTS, POTCAR)
        # ⚠️ 注意: 这里的 POTCAR 必须和你跑 AIMD 时用的一模一样！
        vis = MPRelaxSet(
            struct, 
            user_incar_settings=e0_incar_settings,
            user_potcar_functional="PBE"  # 确保这里也是 PBE
        )
        vis.write_input(task_dir)
        print(f"✅ [成功] {el} -> {task_dir}")
        
    except Exception as e:
        print(f"❌ [失败] {el}: {e}")
        if "POTCAR" in str(e):
            print("   (请确保你的环境中配置了 PMG_VASP_PSP_DIR，或者手动复制 POTCAR)")

🚀 开始生成 E0 输入文件至: Na_Na3SbS4_Data/E0_calcs
✅ [成功] Na -> Na_Na3SbS4_Data/E0_calcs/Na
✅ [成功] Sb -> Na_Na3SbS4_Data/E0_calcs/Sb
✅ [成功] S -> Na_Na3SbS4_Data/E0_calcs/S


/dssg/home/acct-matxzl/matxzl/.conda/envs/dp_qqz/lib/python3.10/site-packages/pymatgen/io/vasp/sets.py:486: BadInputSetWarning: POTCAR data with symbol Na_pv is not known by pymatgen to correspond with the selected user_potcar_functional='PBE'. This POTCAR is known to correspond with functionals ['PBE_54', 'PBE_52']. Please verify that you are using the right POTCARs!
  potcar="\n".join(self.potcar_symbols) if potcar_spec else self.potcar,
/dssg/home/acct-matxzl/matxzl/.conda/envs/dp_qqz/lib/python3.10/site-packages/pymatgen/io/vasp/sets.py:486: BadInputSetWarning: POTCAR data with symbol Sb is not known by pymatgen to correspond with the selected user_potcar_functional='PBE'. This POTCAR is known to correspond with functionals ['PBE_54', 'PBE_52']. Please verify that you are using the right POTCARs!
  potcar="\n".join(self.potcar_symbols) if potcar_spec else self.potcar,
/dssg/home/acct-matxzl/matxzl/.conda/envs/dp_qqz/lib/python3.10/site-packages/pymatgen/io/vasp/sets.py:486: BadInpu

In [2]:
import os
import glob
from dpdispatcher import Machine, Resources, Task, Submission

# ================= 配置 =================
# 指向刚才生成 E0 文件的目录
work_base = "Na_Na3SbS4_Data/E0_calcs" 

# ================= 1. Define Machine =================
machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

# ================= 2. Define Resources =================
# 使用你验证过的成功配置
resources = Resources(
    number_node=1,
    cpu_per_node=64,
    gpu_per_node=0,
    queue_name="64c512g", 
    group_size=3,              # [优化] E0计算很快，把3个任务打包进一个Job，省得排队3次
    module_list=["vasp/6.3.0-intel-2021.4.0"],
    custom_flags=[
        "#SBATCH --partition=64c512g",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn", 
        "#SBATCH --ntasks=64",
        "#SBATCH --time=01:00:00",      # [修改] E0 计算非常快(几秒钟)，1小时绰绰有余
        "#SBATCH --job-name=E0_Calc"    # [修改] 任务名
    ]
)

# 环境变量设置 (保持你成功的配置)
setup_env = (
    "ulimit -s unlimited && "
    "ulimit -l unlimited && "
    "export I_MPI_PMI_LIBRARY=/usr/lib64/libpmi.so && "
    "export I_MPI_FABRICS=shm:ofi && "
    "export I_MPI_PMI=pmi"
)

# ================= 3. Define Command =================
# 使用绝对路径比较稳妥，或者如果你之前的 'vasp_std' 能跑通也可以
# 建议这里填入你之前找到的绝对路径，例如:
# vasp_exe = "/dssg/opt/.../vasp_std" 
# run_cmd = f"mpirun -n 64 {vasp_exe}"

# 如果你确定直接用 vasp_std 没问题：
run_cmd = "mpirun -n 64 vasp_std"

command = f"{setup_env} && {run_cmd}"

# ================= 4. Collect Tasks =================
task_list = []
search_pattern = os.path.join(work_base, "*")

print(f"Scanning {work_base} for tasks...")

for folder_path in glob.glob(search_pattern):
    if os.path.isdir(folder_path):
        folder_name = os.path.basename(folder_path)
        
        # 这里的 backward_files 很重要，我们要读取 OUTCAR 里的能量
        task = Task(
            command=command,
            task_work_path=folder_name,
            forward_files=[], 
            backward_files=['OUTCAR', 'vasprun.xml', 'OSZICAR'] 
        )
        task_list.append(task)

print(f"Found {len(task_list)} tasks.")

# ================= 5. Submit =================
if len(task_list) > 0:
    submission = Submission(
        work_base=work_base,
        machine=machine,
        resources=resources,
        task_list=task_list
    )

    print(f"Submitting {len(task_list)} tasks to Slurm...")
    submission.run_submission()
    print("Submission finished. 请等待任务完成 (通常只需几分钟)。")
else:
    print(f"No subdirectories found in {work_base}")

Scanning Na_Na3SbS4_Data/E0_calcs for tasks...
Found 3 tasks.
Submitting 3 tasks to Slurm...
2025-12-10 10:09:21,867 - INFO : info:check_all_finished: False
2025-12-10 10:09:21,910 - INFO : job: 86d2df369e8ea01ded00972075251977ddb44c57 submit; job_id is 50742010
2025-12-10 10:54:03,010 - INFO : job: 86d2df369e8ea01ded00972075251977ddb44c57 50742010 finished
Submission finished. 请等待任务完成 (通常只需几分钟)。


In [1]:
import os
import glob

# 配置
work_base = "Na_Na3SbS4_Data/E0_calcs_Corrected" 

# 原子序数映射表
atomic_numbers = {
    "Na": 11,
    "S": 16,
    "Sb": 51
}

results = {}

print("📊 正在读取 E0 计算结果...")

for element, z in atomic_numbers.items():
    outcar_path = os.path.join(work_base, element, "OUTCAR")
    
    if not os.path.exists(outcar_path):
        print(f"⚠️ {element}: OUTCAR 未找到 (任务可能还没跑完)")
        continue
        
    energy = None
    try:
        with open(outcar_path, 'r') as f:
            lines = f.readlines()
            # 倒序读取，找最后一次出现的能量
            for line in reversed(lines):
                if "energy without entropy" in line:
                    # 格式通常是: energy without entropy =      -1.31234567  energy(sigma->0) = ...
                    parts = line.split()
                    # 找到等号后面的那个数字
                    energy = float(parts[4]) 
                    break
        
        if energy is not None:
            results[z] = energy
            print(f"✅ {element} (Z={z}): {energy:.5f} eV")
        else:
            print(f"❌ {element}: 未在 OUTCAR 中找到能量数据 (计算可能报错了)")
            
    except Exception as e:
        print(f"❌ {element}: 读取出错 {e}")

# 生成最终字典字符串
if len(results) == 3:
    print("\n🎉 全部成功！请将下面这行复制到你的 MACE 训练脚本中：\n")
    # 格式化输出
    e0_str = "{" + ", ".join([f"{k}: {v:.4f}" for k, v in results.items()]) + "}"
    print(f'E0s = "{e0_str}"')
else:
    print("\n⚠️ 某些元素计算失败，请检查上面的错误信息。")

📊 正在读取 E0 计算结果...
✅ Na (Z=11): -0.22865 eV
✅ S (Z=16): -0.87125 eV
✅ Sb (Z=51): -1.42990 eV

🎉 全部成功！请将下面这行复制到你的 MACE 训练脚本中：

E0s = "{11: -0.2286, 16: -0.8713, 51: -1.4299}"


In [2]:
import os
import glob
import numpy as np
from ase.io import read, write

# ================= 配置区 =================
# 1. 输入路径: 指向你存放 pure Na 和 Na3SbS4 AIMD 结果的目录
# 注意: 这里假设你的目录结构是 Na_Na3SbS4_Data/AIMD_runs/任务名/vasprun.xml
root_dir = "Na_Na3SbS4_Data/AIMD_runs"

# 2. 输出路径: 改个名字，不要叫 LiGa 了
output_dir = "Na_Na3SbS4_Data/dataset"
os.makedirs(output_dir, exist_ok=True) # 自动创建文件夹

output_train = os.path.join(output_dir, "Na_System_train.xyz")
output_valid = os.path.join(output_dir, "Na_System_val.xyz")

validation_ratio = 0.1   # 10% 验证集
shuffle_seed = 42        

# 3. 数据量控制
# 对于纯相数据，数据通常比较简单（重复性高）。
# 如果你跑了 2000 步，这里设为 None (全都要) 或者 2000 都可以。
target_total_frames = None  # None 表示“我不限制数量，有多少用多少”

# ================= 处理逻辑 =================
print(f"🔍 正在扫描 {root_dir} 下的 vasprun.xml ...")
# 递归搜索所有子文件夹
search_pattern = os.path.join(root_dir, "*", "vasprun.xml")
vasp_files = glob.glob(search_pattern)

if not vasp_files:
    print(f"❌ 未找到文件! 请检查路径: {root_dir}")
    # 打印当前目录下的文件夹，帮你排错
    print(f"   当前目录下有: {os.listdir('.')}")
    exit()

all_atoms = []

for i, vasp_file in enumerate(vasp_files):
    folder_name = os.path.basename(os.path.dirname(vasp_file))
    print(f"   [{i+1}/{len(vasp_files)}] 读取: {folder_name} ...", end="\r")
    
    try:
        # 读取轨迹
        # index=':' 读取所有帧; index='::10' 每10帧采一个(稀疏采样)
        traj = read(vasp_file, index=':') 
        
        # 过滤高能异常帧 (可选，防止DeepMD训练跑飞)
        # 简单的做法是跳过前 100 步 equilibration
        # traj = traj[100:] 
        
        all_atoms.extend(traj)
        
    except Exception as e:
        print(f"\n   ⚠️ 读取错误 {folder_name}: {e}")

print(f"\n✅ 数据读取完毕。共收集到 {len(all_atoms)} 帧构型。")

# --- 打乱与拆分 ---
np.random.seed(shuffle_seed)
np.random.shuffle(all_atoms)

# 数量截断
if target_total_frames is not None and len(all_atoms) > target_total_frames:
    print(f"✂️ 截断数据: 从 {len(all_atoms)} 减少到 {target_total_frames}")
    all_atoms = all_atoms[:target_total_frames]
    final_count = target_total_frames
else:
    final_count = len(all_atoms)

# 划分
n_valid = int(final_count * validation_ratio)
n_train = final_count - n_valid
train_set = all_atoms[:n_train]
valid_set = all_atoms[n_train:]

print(f"📊 最终划分: 训练集 {len(train_set)} 帧 | 验证集 {len(valid_set)} 帧")

# --- 保存 ---
print(f"💾 正在写入 {output_train} ...")
write(output_train, train_set, format='extxyz')

print(f"💾 正在写入 {output_valid} ...")
write(output_valid, valid_set, format='extxyz')

print("🎉 数据集生成完成！")

🔍 正在扫描 Na_Na3SbS4_Data/AIMD_runs 下的 vasprun.xml ...
   [6/6] 读取: Na3SbS4_mp-10167 ...toms ...
✅ 数据读取完毕。共收集到 14000 帧构型。
📊 最终划分: 训练集 12600 帧 | 验证集 1400 帧
💾 正在写入 Na_Na3SbS4_Data/dataset/Na_System_train.xyz ...
💾 正在写入 Na_Na3SbS4_Data/dataset/Na_System_val.xyz ...
🎉 数据集生成完成！


In [8]:
import os
from dpdispatcher import Machine, Resources, Task, Submission

# ================= 1. 文件与路径配置 =================

# 定义数据所在的目录 (根据你的要求)
data_dir = "Na_Na3SbS4_Data/dataset"
train_file = os.path.join(data_dir, "Na_System_train.xyz")
valid_file = os.path.join(data_dir, "Na_System_val.xyz")

# 定义训练工作目录 (MACE 的输出会在这里)
work_base = "Na_Na3SbS4_Data/MACE_Training_Run"

# 基础模型 (请确认该文件在当前目录下，或者写绝对路径)
foundation_model = "./mace-mpa-0-medium.model"
job_name = "Na_System_MACE_Train"

# E0 设置: 使用 average 自动拟合，避免 VASP 单原子计算错误
E0s = "average"

# ================= 2. 定义 Machine (保持不变) =================
machine = Machine(
    batch_type="Slurm",
    context_type="LazyLocal",
    local_root="./",
)

# ================= 3. 定义 Resources (⚠️ 重要修改: 切换到 GPU) =================
# 注意：训练神经网络强烈建议使用 GPU。
# 如果你必须用 CPU (64c512g)，训练速度会非常慢。
# 这里假设你切换回了 GPU 分区 (如 a100 或 v100)
resources = Resources(
    number_node=1,
    cpu_per_node=16,          # GPU 节点通常不需要那么多 CPU 核
    gpu_per_node=1,           # ⚠️ 必须申请 GPU
    queue_name="a100",        # ⚠️ 请修改为你超算实际的 GPU 分区名 (例如: a100, gpu, v100)
    group_size=1,
    module_list=[],           # 如果需要加载 cuda 模块，请在这里添加，如 ["cuda/11.8"]
    custom_flags=[
        "#SBATCH --partition=a100", # ⚠️ 对应修改分区名
        "#SBATCH --gres=gpu:1",     # 申请 1 张卡
        "#SBATCH --mem=32G",
        "#SBATCH --time=48:00:00",  # 训练时间较长
        "#SBATCH --job-name=MACE_Train",
        "#SBATCH --mail-type=end",
        "#SBATCH --mail-user=2393474010@sjtu.edu.cn"
    ]
)

# ================= 4. 定义训练命令 =================
# 构建 MACE 训练命令
command_list = [
    "source activate mace2 &&",        # 激活你的 MACE 环境
    "export MPLBACKEND=Agg &&",        # 避免绘图报错
    f"python3 -m mace.cli.run_train \\",
    f"--name='{job_name}' \\",
    f"--foundation_model='{foundation_model}' \\",
    f"--train_file='{train_file}' \\",  # 指向 dataset 目录
    f"--valid_file='{valid_file}' \\",
    f"--E0s='{E0s}' \\",               # 使用 average
    f"--energy_key='energy' \\",
    f"--forces_key='forces' \\",
    f"--stress_key='stress' \\",
    f"--energy_weight=1.0 \\",
    f"--forces_weight=10.0 \\",
    f"--loss='universal' \\",
    f"--lr=0.0005 \\",
    f"--batch_size=10 \\",
    f"--max_num_epochs=100 \\",
    f"--ema \\",
    f"--ema_decay=0.99 \\",
    f"--device=cuda \\",               # 使用 CUDA
    f"--default_dtype='float64' \\",
    f"--checkpoints_dir='checkpoints' \\",
    f"--results_dir='results' \\",
    f"--model_dir='./'"
]

# 将列表拼接成一行命令
command = " ".join(command_list)

# ================= 5. 任务与提交 =================

# 只需要提交一个任务来运行训练脚本
task = Task(
    command=command,
    task_work_path=work_base,
    forward_files=[foundation_model], # 确保把基础模型拷过去
    backward_files=[]
)

submission = Submission(
    work_base=os.getcwd(), # 这里的 work_base 是 dpdispatcher 本地记录路径
    machine=machine,
    resources=resources,
    task_list=[task]
)

print(f"Submitting MACE Training to Slurm...")
print(f"   Train File: {train_file}")
print(f"   Valid File: {valid_file}")
submission.run_submission()
print("Done. 请使用 'squeue' 查看任务状态。")

Submitting MACE Training to Slurm...
   Train File: Na_Na3SbS4_Data/dataset/Na_System_train.xyz
   Valid File: Na_Na3SbS4_Data/dataset/Na_System_val.xyz
2025-12-12 09:49:51,504 - INFO : info:check_all_finished: False
2025-12-12 09:49:51,999 - INFO : job: 0a8901819275397a17cbeceae0ee7ca914e64552 submit; job_id is 50788925
